# Day 029 Project: NotificationBot

## What You're Building

A `NotificationBot` class that:
1. Accepts optional Slack and Discord webhook URLs
2. Generates AI-powered notifications with `build_notification`
3. Exposes `preview(event_type, data)` — builds payloads without sending
4. Exposes `notify(event_type, data)` — builds payloads and POSTs to webhooks

## Project Requirements

1. Implement `NotificationBot` with `preview` and `notify` methods
2. Call `bot.preview(...)` for at least one event and store as `result`
3. Verify with `_run_project_checks()`

Webhook URLs are optional — `preview` works without them. If you have real Slack or Discord webhook URLs, try `notify` too!

In [ ]:
import ollama

## Provided: All Helper Functions

In [ ]:
def format_slack_message(
    title: str,
    body: str,
    color: str = "#36a64f",
) -> dict:
    from datetime import datetime
    return {
        "attachments": [
            {
                "fallback": title,
                "color":    color,
                "title":    title,
                "text":     body,
                "footer":   "NotificationBot",
                "ts":       int(datetime.now().timestamp()),
            }
        ]
    }


def format_discord_embed(
    title: str,
    description: str,
    color: int = 0x00b0f4,
) -> dict:
    return {
        "title":       title,
        "description": description,
        "color":       color,
    }


def truncate_for_chat(text: str, max_chars: int = 2000) -> str:
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."


def ai_summarize_for_chat(
    content: str,
    platform: str = "slack",
    model: str = "llama3.2",
) -> str:
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    f"You are a notification writer for {platform}. "
                    "Write a concise notification summary: under 300 characters, "
                    "no headers, no bullet points. Lead with the most important fact."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Summarise this for a {platform} notification:\n\n"
                    f"{content[:3000]}"
                ),
            },
        ],
    )
    return response["message"]["content"]


def build_notification(
    event_type: str,
    data: dict,
    model: str = "llama3.2",
) -> dict:
    content = (
        f"Event: {event_type}\n\nData:\n"
        + "\n".join(f"  {k}: {v}" for k, v in data.items())
    )
    summary = ai_summarize_for_chat(content, platform="slack", model=model)
    title   = f"[{event_type.upper()}] Notification"
    return {
        "event_type":      event_type,
        "title":           title,
        "summary":         summary,
        "slack_payload":   format_slack_message(title, summary),
        "discord_payload": {
            "embeds": [format_discord_embed(title, truncate_for_chat(summary, 4096))]
        },
    }

## Your Implementation

Implement `NotificationBot` using the helper functions above.

In [ ]:
class NotificationBot:
    def __init__(
        self,
        slack_url: str | None = None,
        discord_url: str | None = None,
        model: str = 'llama3.2',
    ):
        self.slack_url   = slack_url
        self.discord_url = discord_url
        self.model       = model

    def preview(self, event_type: str, data: dict) -> dict:
        # TODO: return build_notification(event_type, data, model=self.model)
        pass

    def notify(self, event_type: str, data: dict) -> dict:
        # TODO: result = self.preview(event_type, data)
        # TODO: result['sent'] = []
        # TODO: if self.slack_url: POST to slack_url; append 'slack:{status}'
        # TODO: if self.discord_url: POST to discord_url; append 'discord:{status}'
        # TODO: return result
        pass

## Events to Test

In [ ]:
# Test with a 'preview' — no webhook URLs needed
# bot = NotificationBot()
# result = bot.preview(
#     event_type='report.generated',
#     data={'rows': 500, 'output': '/tmp/report.xlsx', 'duration_s': 12},
# )
# print(f"Title: {result['title']}")
# print(f"Summary: {result['summary']}")
# print(f"Slack payload keys: {list(result['slack_payload'])}")

# To send to real channels (requires actual webhook URLs):
# bot = NotificationBot(
#     slack_url=os.environ.get('SLACK_WEBHOOK'),
#     discord_url=os.environ.get('DISCORD_WEBHOOK'),
# )
# result = bot.notify('report.generated', {'rows': 500})

## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: NotificationBot has required methods
    try:
        assert 'NotificationBot' in globals()
        for m in ('preview', 'notify'):
            assert hasattr(NotificationBot, m), \
                f'NotificationBot missing method: {m}'
        passed += 1; print('\u2705 Check 1: preview and notify methods present')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: bot is an instance
    try:
        assert 'bot' in globals()
        assert isinstance(bot, NotificationBot)
        passed += 1; print('\u2705 Check 2: bot is a NotificationBot')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: result dict has required keys
    try:
        assert 'result' in globals()
        for k in ('event_type', 'title', 'summary',
                  'slack_payload', 'discord_payload'):
            assert k in result, f"result missing '{k}'"
        passed += 1; print('\u2705 Check 3: result has all required keys')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: slack_payload has attachments
    try:
        assert 'result' in globals()
        assert 'attachments' in result['slack_payload'], \
            f"slack_payload missing attachments: {list(result['slack_payload'])}"
        passed += 1; print('\u2705 Check 4: slack_payload has attachments')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: discord_payload has embeds
    try:
        assert 'result' in globals()
        assert 'embeds' in result['discord_payload'], \
            f"discord_payload missing embeds: {list(result['discord_payload'])}"
        passed += 1; print('\u2705 Check 5: discord_payload has embeds')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `color_for_event(event_type) -> str` helper that returns red ('#e01e5a') for event types ending in '.failed' or '.error', yellow for '.warning', and green for everything else
- Add a `notify_many(events)` method that takes a list of (event_type, data) tuples and sends them all, returning a list of results
- Add Slack Block Kit support: a `format_slack_blocks(title, body, footer)` function that uses the blocks API instead of attachments
- Add Discord fields: extend `format_discord_embed` to accept an optional `fields: list[dict]` parameter for structured key-value display
- Add retry logic: if requests.post raises a ConnectionError, retry up to 3 times with a 2-second delay (Day 31 preview)